# A3.4 · Budgets and stop conditions

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.3 · Egress control](https://spbreed.github.io/cyber-commons/lessons/A3.3.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry |

## What this lesson is

**What it covers.** Run a looping agent against each ceiling and record which one fires first.

**Why a security engineer needs it.** Without a ceiling the loop runs until an external system stops it, and the failure mode is denial of service against yourself. The control it builds is: ceilings bound to the loop, with the run terminating rather than degrading when one is hit.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A budget is what makes "autonomous" a bounded word. Without one, the honest description of the worst case is "until someone notices", and nobody signs off on that when it is written down.

> **At CyberTravels.** What makes “CyberTravels runs autonomously” a bounded sentence: a ceiling on tokens, wall clock, spend and, above all, on how many refunds one run may issue. R1.

## 2 · The framework

```
   bounded run
   +-------------------------------------------+
   | tokens   <= 60k     wall clock <= 5 min   |
   | steps    <= 25      spend     <= $2.00    |
   | actions  <= 3 writes, 0 deletes           |
   +-------------------------------------------+
              |
     hit any ceiling -> stop, report, hand back

   "autonomous" now has a worst case you can write on a page
```

**Mitigates: T4 Resource Overload · T10 Overwhelming Human-in-the-Loop.**

A budget is what makes "autonomous" a bounded word.

A1.13's loop had no exit condition, so it ran until something outside it
intervened. A ceiling turns that into a defined worst case — and a worst case is
the thing you can actually put in a design document, an incident plan or a risk
register.

Four ceilings, because they bound different failures:

**Tokens or cost.** The visible one. Bounds the bill.

**Wall-clock time.** Bounds a workflow step that never returns.

**Actions.** Bounds *consequence*, and it is the one that matters for security.
Twenty tool calls is a very different blast radius from two thousand, whatever
either costs.

**Downstream calls per target.** Bounds harm to other people. A1.13's damage was
not the token spend, it was the capacity taken from everybody else.

Two design rules:

**Terminate, do not degrade.** A loop that hits a ceiling and keeps going with a
smaller model or a shorter context has not been bounded, it has been redirected.

**Make the ceiling visible in the output.** `stopped_by: action_budget` is a
signal to a human that this run is incomplete. Silent truncation is how a
partial result becomes a reported success, which is A1.16 arriving through a
different door.

> **What this control closes.**
>
> Turns an unbounded loop into a defined worst case, and bounds the harm to **other people's** capacity, not just your bill.

## 3 · The check, as a skill

A loop usually carries several budgets and only one of them ever fires. The skill runs A1.13's impossible task against all of them, reports which binds first, and checks what the loop *returns* when it stops — because partial work reported as an answer is a budget converted into a quality problem.

In [ ]:
# skills/runtime/budget-and-stop-condition-audit/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: budget-and-stop-condition-audit
description: >-
  Check that an agent loop has ceilings that actually bind — per-target, token
  and action — and that hitting one returns an incomplete result rather than a
  summary of what it managed. Use when reviewing loop termination, retries, or
  an agent that runs unattended.
allowed-tools: Read, Grep, Glob
---

# The ceiling that binds first is the only one that matters

A loop usually has several budgets and only one of them ever fires. Which one
fires, and how early, is the design; the rest are decoration. The second half
matters more: what the loop **returns** when it stops. A run that halts and
reports its partial work as an answer has converted a budget into a quality
problem.

## When to use this

Any agentic loop, particularly one that retries, and any agent that runs on a
schedule with nobody watching.

## Procedure

**1 — Enumerate every ceiling.** Steps, tokens, wall clock, cost, actions,
per-target attempts. For each, where it is checked and what it does on breach.

**2 — Order them by when they bind.** Run a task that consumes all resources
and record which fires first. A per-target ceiling usually binds long before a
token budget, which means the token budget was never the control.

**3 — Test the breach path.** The result must carry an explicit incomplete
flag. Check what a caller does with it: a loop that returns `complete: False`
into a pipeline that ignores the field has the same outcome as no budget.

**4 — Check the ceiling is not resettable by the agent.** A budget the loop can
extend on its own behalf — by starting a sub-task, spawning a child, or
retrying at a new target — is advisory.

**5 — Derive the numbers from observation.** State the p95 of a legitimate run
and set each ceiling above it. A round number either strangles real work or
never fires.

## Output contract

```json
{
  "ceilings": [{"name": "str", "value": 0, "checked_at": "str", "binds_at_step": 0}],
  "first_to_bind": "str",
  "breach": {"returns_incomplete": true, "caller_respects_flag": false},
  "agent_resettable": ["str"],
  "basis": {"p95_legitimate_run": {"steps": 0, "tokens": 0}}
}
```

## Failure modes

- **Listing budgets without ordering them.** Only the first one exists.
- **Returning partial work as an answer.** The flag is the control; the number
  is the trigger.
- **A ceiling the agent can reset by spawning.** Count the whole tree.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/runtime/budget-and-stop-condition-audit/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/runtime/budget-and-stop-condition-audit/scripts/budget_and_stop_condition_audit.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Run the impossible task against per-target, token and action ceilings and see which one halts it first.

This is the executable half of the `budget-and-stop-condition-audit` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

class Budget:
    def __init__(self, tokens=50_000, seconds=60, actions=20, per_target=5):
        self.limits = {"tokens": tokens, "seconds": seconds,
                       "actions": actions, "per_target": per_target}
        self.used = {"tokens": 0, "seconds": 0, "actions": 0}
        self.targets = {}
    def spend(self, tokens=0, seconds=0, action=None, target=None):
        self.used["tokens"] += tokens
        self.used["seconds"] += seconds
        if action: self.used["actions"] += 1
        if target:
            self.targets[target] = self.targets.get(target, 0) + 1
            if self.targets[target] > self.limits["per_target"]:
                return False, f"per_target ({target})"
        for k in ("tokens", "seconds", "actions"):
            if self.used[k] > self.limits[k]:
                return False, k
        return True, None

def loop(budget):
    """A task that cannot succeed - A1.13's exact scenario, now bounded."""
    steps = 0
    while True:
        steps += 1
        ok, hit = budget.spend(tokens=1800, seconds=0.4, action="query",
                               target="reports-db")
        if not ok:
            return {"steps": steps, "stopped_by": hit, "complete": False}

b = Budget()
r = loop(b)
print(f"steps taken : {r['steps']}")
print(f"stopped by  : {r['stopped_by']}")
print(f"complete    : {r['complete']}   <- visible in the output, not silent")
print()
print(f"{'ceiling':14s}{'limit':>9}{'used':>9}")
for k in ("tokens", "seconds", "actions"):
    print(f"{k:14s}{b.limits[k]:>9}{round(b.used[k], 1):>9}")
print(f"{'per_target':14s}{b.limits['per_target']:>9}{b.targets['reports-db']:>9}")
print()
print("per_target fired first, at 6 calls - long before the token budget or the")
print("action budget. That is the ceiling that protects everyone else, and it is")
print("the one most budgets do not have.")
print()
print("`complete: False` is the other half. A run that stops silently and")
print("reports what it managed becomes A1.16 with extra steps.")
assert r["stopped_by"].startswith("per_target") and not r["complete"]

## What you just proved

The impossible task from A1.13 now stops after six steps, halted by the per-target ceiling — before the token or action budgets are anywhere near exhausted — and the result carries `complete: False` rather than reporting what it managed.

## Your turn

Check whether your agent's budget bounds calls per downstream target. If it only bounds tokens, your cost is protected and the service your agent hammers is not.

---

**Next → [A3.5 · Validating what comes back](https://spbreed.github.io/cyber-commons/lessons/A3.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*